# 03 - QML to C++ signal / slot mapping


## Goal

Use the graph to discover, for each QML component, the C++ backend class registered via `qmlRegisterType`, then walk that class's signals, slots, and properties.


## Prerequisites

- Notebook 02 ran (we rebuild the graph from scratch here regardless).
- Some familiarity with the Qt signal/slot mechanism is helpful.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Rebuild the graph (compact form)


In [ ]:
from src.common.paths import MINI_REPO
from src.repo_ingest.scanner import scan
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id
from src.graph.builder import build_graph

rep = scan(MINI_REPO)
b = ExtractionBundle()
rid = b.add_entity(Entity(id=make_id('Repository', MINI_REPO.name),
                          type='Repository', name=MINI_REPO.name,
                          source_path=str(MINI_REPO)))
for sf in rep.by_kind('cmake'): from_cmake(b, read_cmake(sf.path), rid)
for sf in rep.by_kind('cpp_header') + rep.by_kind('cpp_source'): from_cpp(b, read_cpp(sf.path))
for sf in rep.by_kind('qml'): from_qml(b, read_qml(sf.path))
for sf in rep.by_kind('dbus'): from_dbus(b, read_dbus(sf.path))
for sf in rep.by_kind('kconfig'): from_kconfig(b, read_kconfig(sf.path))
for sf in rep.by_kind('desktop'): from_desktop(b, read_desktop(sf.path))
for sf in rep.by_kind('log'): from_log(b, read_log(sf.path))
g = build_graph(b)
print(f'graph ready: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges')


## 2. Find every QmlComponent and its backend

`qml_backend_for` (in `src/graph/queries.py`) follows the `CONNECTS_TO` edge from a QML component to the `CppClass` it points at. That edge is created during extraction from the `used_types` list in `QmlReadResult` and corresponds to a `qmlRegisterType` call in the C++ source.


In [ ]:
from src.graph.queries import nodes_of_type, qml_backend_for

for qc in nodes_of_type(g, 'QmlComponent'):
    qname = g.nodes[qc]['name']
    backends = qml_backend_for(g, qc)
    if not backends:
        print(f'{qname}: no backend resolved')
        continue
    for b_id in backends:
        b_name = g.nodes[b_id].get('name', '?')
        print(f'{qname:20s} -> {b_name}')


## 3. Walk the backend's signals, slots and properties

Now that we know which C++ class drives each QML view, we can enumerate the things QML code can subscribe to (signals), invoke (slots), or bind to (properties).


In [ ]:
from src.graph.queries import neighbors_by_rel

for qc in nodes_of_type(g, 'QmlComponent'):
    qname = g.nodes[qc]['name']
    for b_id in qml_backend_for(g, qc):
        bname = g.nodes[b_id]['name']
        signals = neighbors_by_rel(g, b_id, 'EMITS')
        slots = neighbors_by_rel(g, b_id, 'HANDLES')
        decls = neighbors_by_rel(g, b_id, 'DECLARES')
        props = [d for d in decls if g.nodes[d].get('type') == 'Property']
        methods = [d for d in decls if g.nodes[d].get('type') == 'Method']
        print(f'\n=== QML {qname} -> C++ {bname} ===')
        print(f'  signals    : {[g.nodes[s]["name"] for s in signals]}')
        print(f'  slots      : {[g.nodes[s]["name"] for s in slots]}')
        print(f'  properties : {[g.nodes[p]["name"] for p in props]}')
        print(f'  methods    : {[g.nodes[m]["name"] for m in methods]}')


## 4. Why this matters for an SLM

A model that can answer *"what C++ class drives `SearchView.qml`?"* without hallucinating needs exactly the edges we walked above. The SFT generator in notebook 06 turns every such pair into a training example.


## Summary

You traced QML components to their C++ backends and listed each backend's signals, slots, and properties. Notebook 04 does the same for D-Bus and KConfig.


## Exercises

1. Add a new QML file under `examples/mini_kde_repo/qml/` that uses a fresh `qmlRegisterType` symbol. Re-run the graph and confirm the new edge appears.
2. Modify `from_qml` in `src/ontology/extractor.py` to also capture QML signals (`signal foo()`). Verify they show up.
3. Write a function `qml_unbacked(g)` that returns QML components with **no** resolved C++ backend. Those are likely smoke tests or design-time stubs.
